In [1]:
"""Competition-shaped end-to-end backend tests for AArch64.

The runner validates compiler -> assembly -> GCC assemble/link -> execution.
QEMU mode is intended for correctness testing only; official performance numbers
must be collected natively on the competition hardware.
"""

from __future__ import annotations

import hashlib
import os
import re
import shutil
import subprocess
import time
from dataclasses import dataclass
from difflib import unified_diff
from pathlib import Path
from typing import Optional, Sequence


class Color:
    GREEN = "\033[92m"
    RED = "\033[91m"
    YELLOW = "\033[93m"
    RESET = "\033[0m"


@dataclass(frozen=True)
class BackendSpec:
    arch: str
    build_dir_name: str
    work_dir_name: str
    gcc_env: str
    gcc_candidates: tuple[str, ...]
    gcc_march: str
    gcc_abi: Optional[str]
    expected_gcc: str
    qemu: str
    qemu_cpu: str
    assembly_banner: str


SPECS = {
    "arm64": BackendSpec(
        arch="arm64",
        build_dir_name="build-arm64-release",
        work_dir_name="backend_test_work_arm64",
        gcc_env="AARCH64_PREFIX",
        gcc_candidates=(
            "aarch64-conda-linux-gnu-",
            str(Path.home() / "miniconda3/envs/LYJYCompiler_test_arm/bin/aarch64-conda-linux-gnu-"),
            "aarch64-linux-gnu-",
            "aarch64-unknown-linux-gnu-",
        ),
        gcc_march="armv8-a",
        gcc_abi=None,
        expected_gcc="11.2.0",
        qemu="qemu-aarch64",
        qemu_cpu="cortex-a53",
        assembly_banner="AArch64",
    ),
}


@dataclass
class BackendTestConfig:
    arch: str
    test_root: Optional[Path] = None
    case_patterns: Sequence[str] = ("**/*.sy",)
    single_cases: Sequence[str] = ()
    max_cases: Optional[int] = None
    stop_on_first_fail: bool = False
    keep_work: bool = False
    quiet_setup: bool = True
    compile_timeout: int = 500
    assemble_timeout: int = 60
    link_timeout: int = 60
    run_timeout: int = 300
    failure_detail_limit: int = 120


@dataclass
class CommandResult:
    returncode: int
    stdout: bytes
    stderr: bytes
    elapsed: float


def discover_project() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "CMakeLists.txt").exists() and (cwd / "runtime").exists():
        return cwd
    if (cwd / "Compiler" / "CMakeLists.txt").exists():
        return cwd / "Compiler"
    return Path("/home/lyjy/Compiler/Compiler").resolve()


class BackendTestRunner:
    def __init__(self, config: BackendTestConfig):
        if config.arch not in SPECS:
            raise ValueError(f"arch must be one of {sorted(SPECS)}, got {config.arch!r}")
        self.config = config
        self.spec = SPECS[config.arch]
        self.project = discover_project()
        self.workspace = self.project.parent
        self.test_root = Path(config.test_root).resolve() if config.test_root else self.project / "testcases"
        self.build_dir = self.project / self.spec.build_dir_name
        # Release CMake targets deliberately emit the official executable name.
        self.compiler = self.build_dir / "compiler"
        self.runtime_c = self.project / "runtime" / "sylib.c"
        self.work_root = self.project / self.spec.work_dir_name
        self.case_dir = self.work_root / "current"
        self.failure_dir = self.work_root / "failures"
        self.report_file = self.work_root / "failure_report.txt"
        self.sylib_obj = self.work_root / "sylib.o"
        self.gcc = self._detect_gcc()
        self.qemu = shutil.which(self.spec.qemu)
        self.results = {name: [] for name in ("AC", "CE", "AE", "LE", "RE", "WA", "TLE", "BAD")}
        self.failure_details: list[tuple[str, str, str]] = []
        self.total_time = 0.0

    def _detect_gcc(self) -> str:
        env_prefix = os.environ.get(self.spec.gcc_env)
        if env_prefix and shutil.which(env_prefix + "gcc"):
            return env_prefix + "gcc"
        for prefix in self.spec.gcc_candidates:
            if shutil.which(prefix + "gcc"):
                return prefix + "gcc"
        raise RuntimeError(
            f"未找到 {self.spec.arch} GCC；请安装 {self.spec.gcc_candidates[0]}gcc "
            f"或设置 {self.spec.gcc_env}。"
        )

    def gcc_arch_args(self) -> list[str]:
        args = [f"-march={self.spec.gcc_march}"]
        if self.spec.gcc_abi:
            args.append(f"-mabi={self.spec.gcc_abi}")
        return args

    def run_command(self, args: Sequence[str], timeout: int, stdin_path: Optional[Path] = None) -> CommandResult:
        stdin_file = stdin_path.open("rb") if stdin_path and stdin_path.exists() else None
        start = time.perf_counter()
        try:
            ret = subprocess.run(
                list(args), cwd=self.workspace, stdin=stdin_file,
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout,
            )
            return CommandResult(ret.returncode, ret.stdout, ret.stderr, time.perf_counter() - start)
        finally:
            if stdin_file:
                stdin_file.close()

    def run_setup(self, args: Sequence[str], stage: str, timeout: int) -> None:
        ret = subprocess.run(
            list(args), cwd=self.workspace, stdout=subprocess.PIPE,
            stderr=subprocess.PIPE, text=True, timeout=timeout,
        )
        if ret.returncode == 0:
            if not self.config.quiet_setup:
                print(ret.stdout, end="")
                print(ret.stderr, end="")
            return
        print(f"{Color.RED}[{stage} failed]{Color.RESET} {' '.join(args)}")
        print((ret.stdout + ret.stderr)[-5000:])
        raise RuntimeError(f"{stage} failed with exit code {ret.returncode}")

    def print_environment(self) -> None:
        gcc_version = subprocess.run(
            [self.gcc, "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, timeout=10,
        ).stdout.splitlines()[0]
        print(f"{Color.YELLOW}[ENV]{Color.RESET} arch={self.spec.arch}, mode=qemu-user")
        print(f"{Color.YELLOW}[ENV]{Color.RESET} gcc={gcc_version}")
        print(f"{Color.YELLOW}[ENV]{Color.RESET} assembler/link flags: {' '.join(self.gcc_arch_args())}")
        if self.spec.expected_gcc not in gcc_version:
            print(
                f"{Color.YELLOW}[WARN]{Color.RESET} 官方 GCC 为 {self.spec.expected_gcc}；"
                "当前版本不能证明官方工具链完全一致。"
            )
        elif "conda-forge" in gcc_version:
            print(
                f"{Color.YELLOW}[WARN]{Color.RESET} GCC 主版本符合 11.2.0，"
                "但当前是 conda-forge 构建，不是官方 Ubuntu 11.2.0-19ubuntu1 包。"
            )
        if not self.qemu:
            raise RuntimeError(f"未找到 {self.spec.qemu}，请安装 qemu-user。")
        qemu_version = subprocess.run(
            [self.qemu, "--version"], stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, timeout=10,
        ).stdout.splitlines()[0]
        print(f"{Color.YELLOW}[ENV]{Color.RESET} qemu={qemu_version}, cpu={self.spec.qemu_cpu}")
        print(f"{Color.YELLOW}[ENV]{Color.RESET} ISA=ARMv8-A AArch64, FP/NEON available on Cortex-A53")
        print(
            f"{Color.YELLOW}[WARN]{Color.RESET} QEMU user-mode 只验证 ISA/FPU/NEON/ABI 与功能；"
            "不能模拟 4 核、isolcpus、L1/L2 缓存、4GB DDR4 或 FPGA 性能。"
        )

    def prepare_runtime(self) -> None:
        if self.work_root.exists() and not self.config.keep_work:
            shutil.rmtree(self.work_root)
        self.case_dir.mkdir(parents=True, exist_ok=True)
        self.failure_dir.mkdir(parents=True, exist_ok=True)
        command = [
            self.gcc, *self.gcc_arch_args(), "-O2", "-w", "-c",
            str(self.runtime_c), "-o", str(self.sylib_obj),
        ]
        self.run_setup(command, "runtime", 120)

    def resolve_case(self, path_text: str) -> Path:
        raw = Path(path_text)
        if raw.is_absolute():
            return raw.resolve()
        for base in (self.test_root, self.project, self.workspace):
            candidate = (base / raw).resolve()
            if candidate.exists():
                return candidate
        return (self.test_root / raw).resolve()

    def collect_cases(self) -> list[Path]:
        if self.config.single_cases:
            cases = [self.resolve_case(path) for path in self.config.single_cases]
        else:
            found = {
                path.resolve()
                for pattern in self.config.case_patterns
                for path in self.test_root.glob(pattern)
                if path.suffix == ".sy"
            }
            cases = sorted(found)
        missing = [path for path in cases if not path.exists()]
        if missing:
            raise RuntimeError(f"测试文件不存在: {missing[0]}")
        return cases[: self.config.max_cases] if self.config.max_cases is not None else cases

    @staticmethod
    def is_performance_case(case: Path) -> bool:
        return any("performance" in part.lower() for part in case.parts)

    def stage_case(self, case: Path) -> tuple[Path, Optional[Path], Path]:
        if self.case_dir.exists():
            shutil.rmtree(self.case_dir)
        self.case_dir.mkdir(parents=True)
        source = self.case_dir / "testcase.sysy"
        shutil.copy2(case, source)
        input_source = case.with_suffix(".in")
        expected_source = case.with_suffix(".out")
        input_file = self.case_dir / "testcase.in" if input_source.exists() else None
        expected_file = self.case_dir / "testcase.out"
        if input_file:
            shutil.copy2(input_source, input_file)
        if expected_source.exists():
            shutil.copy2(expected_source, expected_file)
        return source, input_file, expected_file

    def compile_case(self, source: Path, performance: bool) -> tuple[CommandResult, Path]:
        assembly = self.case_dir / "testcase.s"
        # Exact competition interface: compiler testcase.sysy -S -o testcase.s [-O1].
        command = [str(self.compiler), str(source), "-S", "-o", str(assembly)]
        if performance:
            command.append("-O1")
        return self.run_command(command, self.config.compile_timeout), assembly

    def assemble_case(self, assembly: Path) -> tuple[CommandResult, Path]:
        obj = self.case_dir / "testcase.o"
        # The competition specifies GCC with an explicit -march for assembly.
        command = [self.gcc, *self.gcc_arch_args(), "-c", str(assembly), "-o", str(obj)]
        return self.run_command(command, self.config.assemble_timeout), obj

    def link_case(self, obj: Path) -> tuple[CommandResult, Path]:
        executable = self.case_dir / "testcase"
        command = [self.gcc, *self.gcc_arch_args()]
        command.append("-static")
        command += [str(obj), str(self.sylib_obj), "-o", str(executable)]
        return self.run_command(command, self.config.link_timeout), executable

    def execution_command(self, executable: Path) -> list[str]:
        return [str(self.qemu), "-cpu", self.spec.qemu_cpu, str(executable)]

    @staticmethod
    def parse_runtime(stderr: bytes) -> Optional[float]:
        """Read only timing emitted by runtime/sylib.c."""
        timer_pattern = rb"Timer@\d+-\d+:\s*(\d+)H-(\d+)M-(\d+)S-(\d+)us"
        timers = re.findall(timer_pattern, stderr)
        if timers:
            total_microseconds = sum(
                ((int(h) * 60 + int(m)) * 60 + int(s)) * 1_000_000 + int(us)
                for h, m, s, us in timers
            )
            return total_microseconds / 1_000_000
        total = re.search(rb"TOTAL:\s*(\d+)H-(\d+)M-(\d+)S-(\d+)us", stderr)
        if not total:
            return None
        hours, minutes, seconds, micros = map(int, total.groups())
        return hours * 3600 + minutes * 60 + seconds + micros / 1_000_000

    def run_and_compare(
        self, executable: Path, input_file: Optional[Path], expected: Path
    ) -> tuple[str, float, str]:
        if not expected.exists():
            return "BAD", 0.0, "missing expected .out file"
        ret = self.run_command(self.execution_command(executable), self.config.run_timeout, input_file)
        if ret.returncode < 0:
            detail = f"terminated by signal {-ret.returncode}\n{ret.stderr.decode(errors='replace')[-2000:]}"
            return "RE", 0.0, detail
        runtime = self.parse_runtime(ret.stderr)
        if runtime is None:
            detail = (
                "运行库 stderr 中缺少 Timer/TOTAL；拒绝用主机墙钟时间冒充目标程序时间。\n"
                + ret.stderr.decode("utf-8", errors="replace")[-2000:]
            )
            return "BAD", 0.0, detail
        actual = ret.stdout.decode("utf-8", errors="replace").replace("\r\n", "\n")
        if actual and not actual.endswith("\n"):
            actual += "\n"
        actual += f"{ret.returncode}\n"
        wanted = expected.read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n")
        diff = list(unified_diff(
            actual.splitlines(), wanted.splitlines(),
            fromfile="actual", tofile="expected", lineterm="",
        ))
        if diff:
            return "WA", runtime, "\n".join(diff[: self.config.failure_detail_limit])
        return "AC", runtime, ""

    def record_failure(self, status: str, case: Path, detail: str) -> None:
        try:
            name = str(case.relative_to(self.test_root))
        except ValueError:
            name = str(case)
        self.failure_details.append((status, name, detail.strip()))
        digest = hashlib.sha1(str(case).encode()).hexdigest()[:10]
        failure = self.failure_dir / f"{digest}_{case.stem}"
        failure.mkdir(parents=True, exist_ok=True)
        for path in self.case_dir.iterdir():
            if path.is_file():
                shutil.copy2(path, failure / path.name)

    def print_case(self, status: str, case: Path, runtime: float, index: int, total: int, performance: bool) -> None:
        color = Color.GREEN if status == "AC" else Color.YELLOW if status == "TLE" else Color.RED
        mode = "-O1" if performance else "FUNC"
        name = case.name if len(case.name) <= 25 else case.name[:22] + "..."
        print(
            f"{color}[{status}]{Color.RESET} {name:<25} | {mode:<4} | "
            f"Runtime:{runtime:.6f}s | Total:{self.total_time:>7.2f}s | [{index}/{total}]"
        )

    def test_one(self, case: Path, index: int, total: int) -> str:
        relative = case.relative_to(self.test_root)
        mode = "-O1" if self.is_performance_case(case) else "FUNC"
        print(
            f"{Color.YELLOW}[RUN]{Color.RESET} {relative} | {mode} | "
            f"Total:{self.total_time:>7.2f}s | [{index}/{total}]", flush=True,
        )
        source, input_file, expected = self.stage_case(case)
        performance = self.is_performance_case(case)
        stages = (
            ("CE", self.config.compile_timeout, lambda: self.compile_case(source, performance)),
            ("AE", self.config.assemble_timeout, None),
            ("LE", self.config.link_timeout, None),
        )
        try:
            compile_ret, assembly = stages[0][2]()
            if compile_ret.returncode != 0:
                return self.finish_failure("CE", case, compile_ret.stderr.decode(errors="replace")[-3000:], index, total, performance)
            assemble_ret, obj = self.assemble_case(assembly)
            if assemble_ret.returncode != 0:
                return self.finish_failure("AE", case, assemble_ret.stderr.decode(errors="replace")[-3000:], index, total, performance)
            link_ret, executable = self.link_case(obj)
            if link_ret.returncode != 0:
                return self.finish_failure("LE", case, link_ret.stderr.decode(errors="replace")[-3000:], index, total, performance)
            status, runtime, detail = self.run_and_compare(executable, input_file, expected)
        except subprocess.TimeoutExpired as error:
            # No completed SysY Timer/TOTAL exists on timeout; do not add host wall time to Total.
            return self.finish_failure("TLE", case, f"timeout after {error.timeout}s", index, total, performance)

        self.total_time += runtime
        self.results[status].append(case)
        if status != "AC":
            self.record_failure(status, case, detail)
        self.print_case(status, case, runtime, index, total, performance)
        return status

    def finish_failure(
        self, status: str, case: Path, detail: str, index: int, total: int,
        performance: bool, runtime: float = 0.0,
    ) -> str:
        self.total_time += runtime
        self.results[status].append(case)
        self.record_failure(status, case, detail)
        self.print_case(status, case, runtime, index, total, performance)
        return status

    def write_report(self) -> None:
        lines = []
        for status, case, detail in self.failure_details:
            lines += [f"[{status}] {case}", detail, ""]
        self.report_file.write_text("\n".join(lines) if lines else "No failures.\n", encoding="utf-8")

    def run(self) -> dict[str, list[Path]]:
        self.print_environment()
        if not self.compiler.exists():
            raise RuntimeError(
                f"找不到已构建的 ARM64 Release 编译器：{self.compiler}\n"
                "请先在 VS Code 中运行 ARM64 后端 Release 构建任务。"
            )
        print(f"{Color.YELLOW}[SETUP]{Color.RESET} 使用现有编译器：{self.compiler}")
        self.prepare_runtime()
        cases = self.collect_cases()
        if not cases:
            raise RuntimeError("没有找到任何 .sy 测试用例。")
        print(f"{Color.YELLOW}[SETUP]{Color.RESET} cases={len(cases)}, compiler={self.compiler.name}")
        for index, case in enumerate(cases, 1):
            status = self.test_one(case, index, len(cases))
            if status != "AC" and self.config.stop_on_first_fail:
                break
        self.write_report()
        counts = ", ".join(f"{name}={len(items)}" for name, items in self.results.items() if items)
        print(f"{Color.YELLOW}[SUMMARY]{Color.RESET} {counts or 'no results'}; time={self.total_time:.6f}s")
        print(f"{Color.YELLOW}[REPORT]{Color.RESET} {self.report_file}")
        return self.results


def run_backend_tests(config: BackendTestConfig) -> dict[str, list[Path]]:
    """Run one configured backend suite and return cases grouped by status."""
    return BackendTestRunner(config).run()


# ===== 可直接修改的测试配置 =====
config = BackendTestConfig(
    arch="arm64",
    test_root=discover_project() / "testcases",
    case_patterns=("**/*.sy",),
    single_cases=(),  # 例如 ("functional/00_main.sy",)
    max_cases=None,   # 冒烟测试可改为 1 或 10
    stop_on_first_fail=False,
)

results = run_backend_tests(config)


[ENV] arch=arm64, mode=qemu-user
[ENV] gcc=aarch64-conda-linux-gnu-gcc (conda-forge gcc 11.2.0-16) 11.2.0
[ENV] assembler/link flags: -march=armv8-a
[WARN] GCC 主版本符合 11.2.0，但当前是 conda-forge 构建，不是官方 Ubuntu 11.2.0-19ubuntu1 包。
[ENV] qemu=qemu-aarch64 version 6.2.0 (Debian 1:6.2+dfsg-2ubuntu6.30), cpu=cortex-a53
[ENV] ISA=ARMv8-A AArch64, FP/NEON available on Cortex-A53
[WARN] QEMU user-mode 只验证 ISA/FPU/NEON/ABI 与功能；不能模拟 4 核、isolcpus、L1/L2 缓存、4GB DDR4 或 FPGA 性能。
[SETUP] 使用现有编译器：/home/lyjy/Compiler/Compiler/build-arm64-release/compiler
[SETUP] cases=382, compiler=compiler
[RUN] final/h_performance/2024-0ZE-13.sy | -O1 | Total:   0.00s | [1/382]
[AC] 2024-0ZE-13.sy            | -O1  | Runtime:1.521663s | Total:   1.52s | [1/382]
[RUN] final/h_performance/2024-1WO-11.sy | -O1 | Total:   1.52s | [2/382]


KeyboardInterrupt: 